<div class="alert alert-block alert-success">

# **05 — Statistical & Hypothesis Testing**

## **Goal of this notebook**
This notebook takes the most interesting patterns observed in notebook 04 and tests whether they are statistically real or could have appeared by chance. Every test follows a formal structure:
1. **Hypothesis** — state H₀ (null) and H₁ (alternative)
2. **Assumption check** — normality (Shapiro-Wilk), equal variance (Levene's test)
3. **Choose the right test** — based on data type and assumption results
4. **Run the test** — show the statistic and p-value
5. **Interpret** — plain-English explanation of what the result means, plus effect size

### **Tests in this notebook:**
- Shapiro-Wilk normality tests on key numerical metrics
- Kruskal-Wallis: goals_per_90 across field positions
- Chi-square: foot dominance vs field position association
</div>

In [1]:
# importing the necessaries
import sys
import os
import pandas as pd
import numpy as np
from scipy import stats
# Adding the root to the path to use utils folder
sys.path.append(os.path.abspath(os.path.join('..')))

from utils.db_utils import run_query

import plotly.io as pio
pio.renderers.default = "png"  # drop these 2 lines if you want interactive charts locally

<div class="alert alert-block alert-success">

## **Step 1 — Normality Testing (Shapiro-Wilk)**

Before choosing between parametric (t-test, ANOVA) and non-parametric tests (Mann-Whitney U, Kruskal-Wallis), we need to know whether the key player metrics are normally distributed. Shapiro-Wilk is the most reliable test for this at sample sizes up to ~5000.

**H₀:** The distribution is normal  
**H₁:** The distribution is not normal  

If p < 0.05 → reject H₀ → use non-parametric tests downstream.
</div>

In [2]:
players = run_query("""
    SELECT goals_per_90, assists_per_90, cards_per_90,
           minutes_played_ratio, position, foot
    FROM players
    WHERE goals_per_90 IS NOT NULL
      AND position IS NOT NULL
      AND foot IS NOT NULL
""")

cols_to_test = ['goals_per_90', 'assists_per_90', 'cards_per_90', 'minutes_played_ratio']

normality_results = []
for col in cols_to_test:
    sample = players[col].dropna()
    # Shapiro-Wilk is reliable up to 5000 samples
    if len(sample) > 5000:
        sample = sample.sample(n=5000, random_state=42)
    w_stat, p_val = stats.shapiro(sample)
    normality_results.append({
        'metric': col,
        'n': len(sample),
        'W_statistic': round(w_stat, 4),
        'p_value': round(p_val, 6),
        'is_normal (p>0.05)': p_val > 0.05,
        'test_to_use': 'parametric' if p_val > 0.05 else 'non-parametric'
    })

normality_df = pd.DataFrame(normality_results)
normality_df

,metric,n,W_statistic,p_value,is_normal (p>0.05),test_to_use
0,goals_per_90,5000,0.7591,0.0,False,non-parametric
1,assists_per_90,5000,0.8215,0.0,False,non-parametric
2,cards_per_90,5000,0.8675,0.0,False,non-parametric
3,minutes_played_ratio,5000,0.9616,0.0,False,non-parametric


In [3]:
# Test 1: Do field positions differ significantly in goals_per_90?
# Using Kruskal-Wallis (non-parametric equivalent of one-way ANOVA)
# because normality was rejected above for goals_per_90
#
# H₀: goals_per_90 is the same across all positions
# H₁: at least one position has a significantly different goals_per_90

position_groups = [
    players.loc[players['position'] == pos, 'goals_per_90'].dropna().values
    for pos in players['position'].unique()
    if len(players.loc[players['position'] == pos, 'goals_per_90'].dropna()) >= 10
]
positions_included = [
    pos for pos in players['position'].unique()
    if len(players.loc[players['position'] == pos, 'goals_per_90'].dropna()) >= 10
]

h_stat, p_val = stats.kruskal(*position_groups)

# Effect size: eta-squared (η²) = (H - k + 1) / (n - k)
n_total = sum(len(g) for g in position_groups)
k = len(position_groups)
eta_squared = (h_stat - k + 1) / (n_total - k)

print(f"Positions tested: {positions_included}")
print(f"Kruskal-Wallis H = {h_stat:.4f}, p = {p_val:.2e}")
print(f"Effect size (η²) = {eta_squared:.4f}")
print(f"Decision: {'Reject H₀' if p_val < 0.05 else 'Fail to reject H₀'}")
print()

# Median goals_per_90 per position for interpretation
players.groupby('position')['goals_per_90'].agg(['median', 'mean', 'count']).round(4)

Positions tested: ['Goalkeeper', 'Attack', 'Midfield', 'Defender']
Kruskal-Wallis H = 3721.9328, p = 0.00e+00
Effect size (η²) = 0.3779
Decision: Reject H₀



,median,mean,count
position,,,
Attack,0.23,0.2528,2770
Defender,0.03,0.0391,3412
Goalkeeper,0.00,0.0001,814
Midfield,0.07,0.0981,2848


In [4]:
# Test 2: Is there a significant association between foot dominance and field position?
# Using Chi-square test of independence (both variables are categorical)
#
# H₀: foot dominance and field position are independent
# H₁: foot dominance and field position are associated

# filter to left/right only — 'both' has too few observations to be meaningful
foot_pos = players[players['foot'].isin(['left', 'right'])]

contingency = pd.crosstab(foot_pos['position'], foot_pos['foot'])

chi2_stat, p_val, dof, expected = stats.chi2_contingency(contingency)

# Effect size: Cramér's V
n = contingency.values.sum()
cramer_v = np.sqrt(chi2_stat / (n * (min(contingency.shape) - 1)))

print(f"Chi-square = {chi2_stat:.4f}, dof = {dof}, p = {p_val:.2e}")
print(f"Effect size (Cramér's V) = {cramer_v:.4f}  "
      f"({'small' if cramer_v < 0.1 else 'medium' if cramer_v < 0.3 else 'large'})")
print(f"Decision: {'Reject H₀' if p_val < 0.05 else 'Fail to reject H₀'}")
print()

# Proportions table for interpretation
contingency.div(contingency.sum(axis=1), axis=0).round(3)

Chi-square = 280.6087, dof = 3, p = 1.56e-60
Effect size (Cramér's V) = 0.1720  (medium)
Decision: Reject H₀



foot,left,right
position,,
Attack,0.238,0.762
Defender,0.364,0.636
Goalkeeper,0.141,0.859
Midfield,0.211,0.789


<div class="alert alert-block alert-success">

## **Findings Summary**

### Normality
All four player metrics (goals_per_90, assists_per_90, cards_per_90, minutes_played_ratio) are **non-normally distributed** (Shapiro-Wilk p < 0.05 for all). This is expected — football performance metrics are heavily right-skewed: most players score zero goals in most games. **All subsequent group comparisons in this notebook use non-parametric tests.**

### Test 1 — Kruskal-Wallis: goals_per_90 by position
The test result confirms whether positions genuinely differ in goal contribution rate. The result is expected to be significant (Attackers score more than Defenders) but effect size tells us how *large* this difference is — a small η² would suggest the positional distinction is less analytically useful than it appears.

### Test 2 — Chi-square: foot dominance vs position
If association is found (p < 0.05) it suggests that certain positions attract players with a specific dominant foot — e.g., left-backs disproportionately left-footed. Cramér's V quantifies how strong this structural pattern is across the dataset.

**Both findings carry forward as confirmed (or rejected) hypotheses into notebook 06 — KPI Design & Analysis.**
</div>